## Aviso de versión y resultados vigentes

Notebook histórico conservado como insumo. Este archivo puede incluir celdas y resultados anteriores; no uses sus cifras como evidencia final ni lo ejecutes de principio a fin. El corte vigente excluye 2020. Consulta Procesos_revisado.docx, README.md y las salidas recalculadas en la raíz del proyecto. AWS/Grafana no se verificaron en esta ejecución; no ejecutes cargas con credenciales.


> **Decisión metodológica:** 2020 se excluye de todo análisis, modelo, evaluación y visualización. El CSV original se conserva intacto. Los resultados guardados en este notebook se generaron antes de aplicar el filtro y son obsoletos; no deben citarse. Reejecute el flujo con `Year <> 2020` antes de usar resultados. Consulte `DECISION_EXCLUSION_2020.md`.

# Análisis exploratorio de datos (EDA): vuelos comerciales en EE. UU., 2018–2022

**Curso:** Herramientas de Big Data · **Proyecto final**

**Dataset:** [Flight Status Prediction (Kaggle)](https://www.kaggle.com/datasets/robikscube/flight-delay-dataset-20182022). Datos del *Marketing Carrier On-Time Performance* del Bureau of Transportation Statistics (BTS) de EE. UU. Son 5 archivos CSV (uno por año) con 61 columnas cada uno.

**Objetivo de este notebook:** unir los consolidados anuales en un único archivo parquet, revisar la calidad de los datos y dejar definida una base de análisis confiable para estudiar retrasos y cancelaciones.

## Contenido
1. Preparación del entorno
2. Unión de los 5 años en un solo parquet
3. Verificación de la unión
4. Primera mirada a los datos: estructura y tipos
5. Valores nulos
6. Origen de los nulos: vuelos cancelados y desviados
7. Columnas disponibles y cobertura por mes
8. Vuelos, cancelaciones y desvíos por año
9. Base de análisis: vuelos operados
10. Retrasos por año
11. Diagnóstico: ¿por qué 2018 tiene menos vuelos?
12. Base comparable: enero a julio
13. Conclusiones parciales y próximos pasos

In [ ]:
raise RuntimeError("Notebook histórico: contiene código anterior a la exclusión de 2020. No ejecutar esta copia; usar el notebook principal de la carpeta raíz.")


## 1. Preparación del entorno

**Por qué:** los CSV pesan en total unos **10 GB**, demasiado para cargarlos completos en pandas sin riesgo de saturar la memoria. Usamos dos librerías:

- **DuckDB:** motor de consultas SQL que lee archivos grandes sin cargarlos completos en memoria. Con él unimos los CSV y hacemos las agregaciones.
- **PyArrow:** necesaria para escribir y leer el formato **parquet**, un formato columnar y comprimido. Permite leer solo las columnas que se necesitan, así que las consultas son mucho más rápidas que con CSV.

> Si ya las tienes instaladas, puedes saltarte la primera celda. Si las instalas aquí, reinicia el kernel después.

In [ ]:
%pip install duckdb pyarrow

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 70)   # mostrar todas las columnas
pd.set_option("display.max_rows", 100)     # mostrar hasta 100 filas

# Carpeta con los CSV: "." significa la misma carpeta donde está este notebook
carpeta_datos = Path(".")

# Archivo parquet que vamos a crear con los 5 años unidos
archivo_final = Path("Combined_Flights_2018_2022.parquet")
ruta = str(archivo_final)   # DuckDB recibe la ruta como texto

Antes de unir, confirmamos que Python encuentra los 5 archivos CSV y vemos su tamaño. Si esta lista sale vacía, hay un problema con la ruta (por ejemplo, el notebook no está en la misma carpeta que los datos).

In [ ]:
archivos_csv = sorted(p for p in carpeta_datos.glob("Combined_Flights_20??.csv") if "2020" not in p.name)

for archivo in archivos_csv:
    print(f"{archivo.name:<30} {archivo.stat().st_size / 1e6:>8,.0f} MB")

assert len(archivos_csv) == 4, "Se esperan los CSV de 2018, 2019, 2021 y 2022; 2020 se excluye."

**Resultado:** los cinco archivos pesan entre 1.4 y 2.8 GB cada uno (≈ 10.2 GB en total). Este tamaño justifica el uso de DuckDB y parquet.

## 2. Unión de los 5 años en un solo parquet

**Por qué:** trabajar con un único archivo evita repetir la lectura de 10 GB de CSV cada vez que hagamos una consulta. La instrucción hace tres cosas a la vez:

- `Combined_Flights_20??.csv` toma los cinco archivos (los `??` son los dos últimos dígitos del año) y los trata como una sola tabla.
- `union_by_name = true` une las columnas por su **nombre**, no por su posición, lo que protege contra diferencias de orden entre años.
- `sample_size = -1` hace que DuckDB revise **todas** las filas para decidir el tipo de cada columna. Es más lento, pero evita errores de tipo en columnas con valores raros.

Se guarda con compresión **ZSTD** para reducir el tamaño en disco. La celda solo crea el archivo si todavía no existe, así que puedes volver a ejecutar el notebook sin repetir este paso, que tarda varios minutos.

In [ ]:
if archivo_final.exists():
    print(f"El parquet ya existe ({archivo_final.stat().st_size / 1e9:.1f} GB). No se vuelve a crear.")
else:
    duckdb.sql(f"""
        COPY (
            SELECT *
            FROM read_csv('Combined_Flights_20??.csv', union_by_name = true, sample_size = -1)
        )
        TO '{archivo_final}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    print("Listo:", archivo_final, f"({archivo_final.stat().st_size / 1e9:.1f} GB)")

**Resultado:** el parquet resultante pesa **0.8 GB**, frente a ≈ 10.2 GB de los CSV originales: unas 12 veces menos espacio, con los mismos datos.

## 3. Verificación de la unión

**Por qué:** antes de analizar, hay que comprobar que la unión salió bien: que estén las **61 columnas** y que cada año aporte un número razonable de vuelos.

In [ ]:
esquema = pq.read_schema(archivo_final)
print("Columnas:", len(esquema.names))

total = duckdb.sql(f"SELECT COUNT(*) FROM '{ruta}'").fetchone()[0]
print(f"Total de vuelos: {total:,}")

duckdb.sql(f"""
    SELECT Year, COUNT(*) AS vuelos
    FROM '{ruta}'
    GROUP BY Year
    ORDER BY Year
""").df()

**Análisis:**

- Se conservaron las **61 columnas** y hay **29,193,782 vuelos** en total.
- Los cinco años están presentes, pero el volumen es **irregular**: 2018 tiene 5.7 millones de vuelos, 2019 tiene 8.1 millones y 2022 solo 4.1 millones.
- Es poco probable que el tráfico aéreo cambie un 40% entre 2018 y 2019. Esto es una **señal de alerta**, que revisamos en las secciones 7 y 11 antes de comparar años.

## 4. Primera mirada a los datos: estructura y tipos

**Por qué:** ver algunas filas y el tipo de cada columna ayuda a entender qué representa cada registro (un vuelo programado en una fecha) y a detectar columnas que habrá que transformar.

In [ ]:
duckdb.sql(f"SELECT * FROM '{ruta}' LIMIT 5").df()

In [ ]:
duckdb.sql(f"DESCRIBE SELECT * FROM '{ruta}'").df()[["column_name", "column_type"]]

**Análisis:**

- Cada fila es **un vuelo en una fecha concreta** (aerolínea, origen, destino, hora programada y real, retrasos, distancia, etc.).
- `FlightDate` ya viene como fecha y `Cancelled` / `Diverted` como booleanos, lo que facilita los filtros.
- Las horas (`CRSDepTime`, `DepTime`, `ArrTime`) son números con formato **hhmm** (por ejemplo, `1202` significa 12:02). Para analizar por hora del día habrá que convertirlas.
- Muchas columnas numéricas son **identificadores** (`DOT_ID_...`, `OriginAirportID`, `OriginWac`, etc.), no mediciones. No tiene sentido promediarlas ni correlacionarlas.
- Hay varias columnas que describen lo mismo con distinto nombre (por ejemplo, tres versiones del código de aerolínea). Más adelante escogeremos una para los análisis por aerolínea.

## 5. Valores nulos

**Por qué:** los datos faltantes pueden sesgar los promedios y los porcentajes. Antes de decidir qué hacer con ellos hay que saber **dónde están** y **cuántos son**. `SUMMARIZE` calcula en una sola consulta el porcentaje de nulos, los valores únicos y el rango de cada columna.

In [ ]:
resumen = duckdb.sql(f"SUMMARIZE SELECT * FROM '{ruta}'").df()

resumen[["column_name", "column_type", "null_percentage", "approx_unique", "min", "max"]]

In [ ]:
# Solo las columnas que tienen algún nulo, ordenadas de más a menos
resumen[resumen["null_percentage"] > 0] \
    .sort_values("null_percentage", ascending=False) \
    [["column_name", "null_percentage"]]

**Análisis:**

- Solo **17 de las 61 columnas** tienen valores nulos, y el máximo es **2.92%** (`AirTime`). La calidad general es alta.
- Los nulos se agrupan en dos bloques con porcentajes casi idénticos:
  - **≈ 2.6%** en las columnas de la salida (`DepTime`, `DepDelay`, `DepDel15`, `TaxiOut`, `WheelsOff`).
  - **≈ 2.7 – 2.9%** en las de la llegada (`ArrTime`, `ArrDelay`, `ArrDel15`, `TaxiIn`, `WheelsOn`, `AirTime`).
- Que los nulos se muevan **en bloque** sugiere que no son errores de captura, sino vuelos que nunca completaron alguna etapa. Lo comprobamos en la siguiente sección.
- `Tail_Number` (matrícula del avión) tiene 0.92% de nulos, un caso aparte.

## 6. Origen de los nulos: vuelos cancelados y desviados

**Por qué:** si los nulos corresponden a vuelos que no despegaron o no llegaron a su destino, **no se deben imputar ni eliminar como "datos malos"**: son información válida (el vuelo no ocurrió como estaba planeado). Lo verificamos cruzando los nulos con las columnas `Cancelled` y `Diverted`.

In [ ]:
duckdb.sql(f"""
    SELECT
        Cancelled,
        Diverted,
        COUNT(*) AS vuelos,
        ROUND(100 * AVG(CAST(DepDelay IS NULL AS INT)), 2) AS pct_sin_DepDelay,
        ROUND(100 * AVG(CAST(ArrDelay IS NULL AS INT)), 2) AS pct_sin_ArrDelay
    FROM '{ruta}'
    GROUP BY Cancelled, Diverted
    ORDER BY Cancelled, Diverted
""").df()

**Análisis:**

| Grupo | Vuelos | % del total | Sin `DepDelay` | Sin `ArrDelay` |
|---|---|---|---|---|
| Operados (ni cancelados ni desviados) | 28,348,168 | 97.1% | 0.00% | 0.0% |
| Desviados | 68,347 | 0.23% | 0.01% | 100% |
| Cancelados | 777,265 | 2.66% | 98.03% | 100% |
| Cancelados y desviados | 2 | ~0% | 100% | 100% |

- Los vuelos operados prácticamente **no tienen nulos**: los vacíos se concentran en cancelados y desviados.
- Los **desviados** sí despegaron (casi todos tienen `DepDelay`), pero no tienen retraso de llegada porque aterrizaron en otro aeropuerto.
- Los **cancelados** casi nunca tienen datos de salida. Un 2% sí los tiene, probablemente vuelos que salieron del gate y se cancelaron después.
- **Decisión:** no se imputan los nulos. Para estudiar retrasos usaremos solo los vuelos operados (sección 9).

## 7. Columnas disponibles y cobertura por mes

**Por qué:** el archivo de documentación (`readme.html`) describe unas 120 columnas del BTS, pero la versión de Kaggle trae solo 61. Conviene saber qué columnas importantes **no** están, porque limitan las preguntas que podemos responder. Además, la irregularidad de vuelos por año vista en la sección 3 obliga a revisar **qué meses cubre cada año**, porque comparar un año completo con uno incompleto distorsiona los porcentajes (la estacionalidad pesa mucho en el transporte aéreo).

In [ ]:
columnas = duckdb.sql(f"DESCRIBE SELECT * FROM '{ruta}'").df()["column_name"].tolist()

for c in ["CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay", "Div1Airport", "CancellationCode"]:
    print(f"{c:<20} {'está' if c in columnas else 'NO está'}")

In [ ]:
cobertura = duckdb.sql(f"""
    SELECT Year, Month, COUNT(*) AS vuelos
    FROM '{ruta}'
    GROUP BY Year, Month
""").df()

tabla_cobertura = cobertura.pivot(index="Year", columns="Month", values="vuelos")

# Tabla: 0 significa "sin datos para ese mes"
tabla_cobertura.fillna(0).astype(int)

In [ ]:
# Mismo dato en gráfico: vuelos por mes, una línea por año
tabla_cobertura.T.plot(marker="o", figsize=(10, 5))
plt.title("Vuelos por mes y año")
plt.xlabel("Mes")
plt.ylabel("Número de vuelos")
plt.xticks(range(1, 13))
plt.legend(title="Año")
plt.grid(alpha=0.3)
plt.show()

Texto histórico retirado: incluía análisis o cifras de 2020. El año está excluido del proyecto y no se presentan resultados sobre él. Esta copia es de referencia; utilice el notebook principal para recalcular los resultados sin 2020.

## 8. Vuelos, cancelaciones y desvíos por año

**Por qué:** las cancelaciones y los desvíos son otra forma de mala puntualidad, distinta del retraso. Medimos qué proporción de los vuelos programados no se completó cada año.

In [ ]:
por_anio = duckdb.sql(f"""
    SELECT
        Year,
        COUNT(*)                                        AS vuelos,
        ROUND(100 * AVG(CAST(Cancelled AS INT)), 2)     AS pct_cancelados,
        ROUND(100 * AVG(CAST(Diverted  AS INT)), 2)     AS pct_desviados
    FROM '{ruta}'
    GROUP BY Year
    ORDER BY Year
""").df()

por_anio

In [ ]:
por_anio.set_index("Year")["pct_cancelados"].plot(kind="bar", figsize=(7, 4), color="#d1495b")
plt.title("% de vuelos cancelados por año (todos los meses disponibles)")
plt.ylabel("%")
plt.xlabel("")
plt.xticks(rotation=0)
plt.show()

Texto histórico retirado: incluía análisis o cifras de 2020. El año está excluido del proyecto y no se presentan resultados sobre él. Esta copia es de referencia; utilice el notebook principal para recalcular los resultados sin 2020.

## 9. Base de análisis: vuelos operados

**Por qué:** para medir retrasos solo tienen sentido los vuelos que **despegaron y llegaron a su destino**. Un vuelo cancelado no tiene retraso de llegada, y uno desviado no aterrizó donde estaba programado. Creamos una **vista** de DuckDB: no duplica los datos, solo guarda el filtro, y cada consulta posterior la usa como si fuera una tabla.

In [ ]:
duckdb.sql(f"""
    CREATE OR REPLACE VIEW vuelos_operados AS
    SELECT *
    FROM '{ruta}'
    WHERE Cancelled = FALSE AND Diverted = FALSE
""")

duckdb.sql("""
    SELECT
        COUNT(*)                            AS vuelos_operados,
        SUM(CAST(ArrDelay IS NULL AS INT))  AS sin_ArrDelay
    FROM vuelos_operados
""").df()

**Análisis:**

- La base de retrasos tiene **28,348,168 vuelos**, el 97.1% del total.
- Solo **569 vuelos operados** no tienen `ArrDelay`, un 0.002%. Es insignificante y las funciones de promedio los ignoran automáticamente, así que no hace falta tratarlos.
- A partir de aquí, **los análisis de retrasos usan `vuelos_operados`** y los de cancelaciones usan el parquet completo.

## 10. Retrasos por año

**Por qué:** es la primera medida de puntualidad. Usamos dos indicadores:

- **Retraso medio de llegada** en minutos (los valores negativos significan llegar antes de lo programado).
- **% de vuelos con 15 minutos o más de retraso** (`ArrDel15`), la definición oficial de "vuelo retrasado" en EE. UU.

In [ ]:
retrasos_anio = duckdb.sql("""
    SELECT
        Year,
        COUNT(*)                          AS vuelos,
        ROUND(AVG(ArrDelay), 1)           AS retraso_medio_min,
        ROUND(100 * AVG(ArrDel15), 1)     AS pct_retrasados_15min
    FROM vuelos_operados
    GROUP BY Year
    ORDER BY Year
""").df()

retrasos_anio

Texto histórico retirado: incluía análisis o cifras de 2020. El año está excluido del proyecto y no se presentan resultados sobre él. Esta copia es de referencia; utilice el notebook principal para recalcular los resultados sin 2020.

## 11. Diagnóstico: ¿por qué 2018 tiene menos vuelos?

**Por qué:** de enero a agosto de 2018 hay entre 28% y 54% menos vuelos que en esos mismos meses de 2019. Si el hueco se concentra en una o dos aerolíneas, entenderemos la causa y sabremos si los porcentajes de 2018 son confiables. Si afecta a todas por igual, el faltante es más bien general.

La consulta compara, aerolínea por aerolínea, el número de vuelos de enero a agosto de 2018 y de 2019. La columna `razon_2018_vs_2019` vale 1 si no hubo cambio, 0.5 si 2018 tiene la mitad, y vacía si la aerolínea no existía en 2019.

In [ ]:
duckdb.sql(f"""
    SELECT
        Airline,
        SUM(CASE WHEN Year = 2018 THEN 1 ELSE 0 END)  AS vuelos_2018,
        SUM(CASE WHEN Year = 2019 THEN 1 ELSE 0 END)  AS vuelos_2019,
        ROUND(SUM(CASE WHEN Year = 2018 THEN 1 ELSE 0 END)
              / NULLIF(SUM(CASE WHEN Year = 2019 THEN 1 ELSE 0 END), 0), 2) AS razon_2018_vs_2019
    FROM '{ruta}'
    WHERE Month <= 8 AND Year IN (2018, 2019)
    GROUP BY Airline
    ORDER BY vuelos_2019 DESC
""").df()

**Análisis** *(completa esta celda después de ejecutar la anterior)*:

Qué buscar en la tabla:
- Aerolíneas con `razon_2018_vs_2019` cercana a 0 o con `vuelos_2018 = 0`: el faltante se concentra en ellas.
- Si casi todas tienen razones entre 0.4 y 0.8: el recorte es general y no depende de la aerolínea.

Escribe aquí tus observaciones y la decisión tomada (por ejemplo, "excluir 2018 de las comparaciones de volumen" o "usar solo septiembre – diciembre de 2018").

## 12. Base comparable: enero a julio

**Por qué:** los únicos meses que existen en **los cinco años** son enero a julio (2022 termina en julio). Si limitamos el análisis a esos meses, cada año se compara contra el mismo periodo del calendario y la estacionalidad deja de distorsionar los resultados. Creamos una nueva vista con esa restricción y repetimos las medidas de las secciones 8 y 10.

In [ ]:
duckdb.sql("""
    CREATE OR REPLACE VIEW vuelos_comparables AS
    SELECT *
    FROM vuelos_operados
    WHERE Month <= 7
""")

retrasos_comparables = duckdb.sql("""
    SELECT
        Year,
        COUNT(*)                          AS vuelos,
        ROUND(AVG(ArrDelay), 1)           AS retraso_medio_min,
        ROUND(100 * AVG(ArrDel15), 1)     AS pct_retrasados_15min
    FROM vuelos_comparables
    GROUP BY Year
    ORDER BY Year
""").df()

retrasos_comparables

In [ ]:
cancelaciones_comparables = duckdb.sql(f"""
    SELECT
        Year,
        COUNT(*)                                        AS vuelos,
        ROUND(100 * AVG(CAST(Cancelled AS INT)), 2)     AS pct_cancelados
    FROM '{ruta}'
    WHERE Month <= 7
    GROUP BY Year
    ORDER BY Year
""").df()

cancelaciones_comparables

In [ ]:
# Comparación gráfica: todos los meses disponibles vs solo enero–julio
comparacion = retrasos_anio.set_index("Year")[["pct_retrasados_15min"]] \
    .join(retrasos_comparables.set_index("Year")[["pct_retrasados_15min"]],
          lsuffix="_todos_los_meses", rsuffix="_ene_jul")

comparacion.plot(kind="bar", figsize=(8, 4))
plt.title("% de vuelos retrasados: todos los meses vs enero–julio")
plt.ylabel("%")
plt.xlabel("")
plt.xticks(rotation=0)
plt.legend(["Todos los meses disponibles", "Solo enero–julio"])
plt.show()

Texto histórico retirado: incluía análisis o cifras de 2020. El año está excluido del proyecto y no se presentan resultados sobre él. Esta copia es de referencia; utilice el notebook principal para recalcular los resultados sin 2020.

Texto histórico retirado: incluía análisis o cifras de 2020. El año está excluido del proyecto y no se presentan resultados sobre él. Esta copia es de referencia; utilice el notebook principal para recalcular los resultados sin 2020.